In [ ]:
# In[1]:

import pandas as pd
import numpy as np

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (df_metric, df_trace, df_log, df_error):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# --- metric.csv aggregates ---
metric_general = pd.Series({
    'total_rows': len(df_metric),
    'earliest_timestamp': df_metric['timestamp'].min(),
    'latest_timestamp': df_metric['timestamp'].max()
})

# distinct cmdb_id and kpi_name (limit top 50)
metric_cmdb_list = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()))[:50]
metric_kpi_list = pd.Series(sorted(df_metric['kpi_name'].dropna().unique()))[:50]

# grouped table: (cmdb_id, kpi_name, count, global P95 of value), top 20 by count
metric_grouped = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
)
metric_grouped_top20 = metric_grouped.head(20)

# --- trace.csv aggregates ---
trace_general = pd.Series({
    'total_rows': len(df_trace),
    'earliest_timestamp': df_trace['timestamp'].min(),
    'latest_timestamp': df_trace['timestamp'].max()
})

trace_cmdb_list = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()))[:50]
trace_name_list = pd.Series(sorted(df_trace['trace_name'].dropna().unique()))[:50]

trace_grouped = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
)
trace_grouped_top20 = trace_grouped.head(20)

# --- log.csv aggregates ---
log_general = pd.Series({
    'total_rows': len(df_log),
    'earliest_timestamp': df_log['timestamp'].min(),
    'latest_timestamp': df_log['timestamp'].max()
})

log_cmdb_list = pd.Series(sorted(df_log['cmdb_id'].dropna().unique()))[:50]
log_name_list = pd.Series(sorted(df_log['log_name'].dropna().unique()))[:50]

log_grouped = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(count='count', p95=lambda x: float(x.quantile(0.95)))
    .reset_index()
    .sort_values('count', ascending=False)
)
log_grouped_top20 = log_grouped.head(20)

# --- error_logs.csv aggregates ---
error_total_rows = len(df_error)
error_cmdb_counts_top20 = df_error['cmdb_id'].value_counts().head(20)
error_sample_rows = df_error.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(5)

# Display compact summaries (variables below will be shown by the IPython kernel)
metric_general, metric_cmdb_list, metric_kpi_list, metric_grouped_top20, \
trace_general, trace_cmdb_list, trace_name_list, trace_grouped_top20, \
log_general, log_cmdb_list, log_name_list, log_grouped_top20, \
error_total_rows, error_cmdb_counts_top20, error_sample_rows

```
Out[1]:
```
```python
# Summarize the previously computed aggregates in plain English.
metric_time_range = f"{metric_general['earliest_timestamp']} to {metric_general['latest_timestamp']}"
trace_time_range = f"{trace_general['earliest_timestamp']} to {trace_general['latest_timestamp']}"
log_time_range = f"{log_general['earliest_timestamp']} to {log_general['latest_timestamp']}"

metric_cmdbs = ", ".join(metric_cmdb_list.tolist())
metric_kpis = ", ".join(metric_kpi_list.tolist())

# pick top 3 cmdb_id,kpi_name combos from metric_grouped_top20 for short mention
metric_top_samples = metric_grouped_top20[['cmdb_id','kpi_name','count','p95']].head(3)
metric_top_text = "; ".join(
    f"{row.cmdb_id}/{row.kpi_name} (count={int(row['count'])}, p95={row['p95']:.6g})"
    for _, row in metric_top_samples.iterrows()
)

trace_cmdbs = ", ".join(trace_cmdb_list.tolist())
trace_names_preview = ", ".join(trace_name_list.tolist()[:10]) + (", ..." if len(trace_name_list) > 10 else "")
trace_top_sample = trace_grouped_top20[['cmdb_id','trace_name','count','p95']].head(3)
trace_top_text = "; ".join(
    f"{row.cmdb_id}/{row.trace_name} (count={int(row['count'])}, p95={row['p95']:.6g})"
    for _, row in trace_top_sample.iterrows()
)

log_cmdbs = ", ".join(log_cmdb_list.tolist())
log_names = ", ".join(log_name_list.tolist())
log_top_sample = log_grouped_top20[['cmdb_id','log_name','count','p95']].head(3)
log_top_text = "; ".join(
    f"{row.cmdb_id}/{row.log_name} (count={int(row['count'])}, p95={row['p95']:.6g})"
    for _, row in log_top_sample.iterrows()
)

error_total = int(error_total_rows)

summary = (
    "Telemetry aggregate summary (plain English):\n\n"
    f"1) metric.csv: {int(metric_general['total_rows'])} rows, timeframe {metric_time_range} (UTC).\n"
    f"   Distinct services (cmdb_id): {metric_cmdbs}.\n"
    f"   Distinct KPI names: {metric_kpis}.\n"
    f"   Top metric groups samples: {metric_top_text}.\n\n"
    f"2) trace.csv: {int(trace_general['total_rows'])} rows, timeframe {trace_time_range} (UTC).\n"
    f"   Distinct services (cmdb_id): {trace_cmdbs}.\n"
    f"   Example trace names (first 10 shown): {trace_names_preview}.\n"
    f"   Top trace groups samples: {trace_top_text}.\n\n"
    f"3) log.csv: {int(log_general['total_rows'])} rows, timeframe {log_time_range} (UTC).\n"
    f"   Distinct services (cmdb_id): {log_cmdbs}.\n"
    f"   Log metric names: {log_names}.\n"
    f"   Top log groups samples: {log_top_text}.\n\n"
    f"4) error_logs.csv: {error_total} rows (no error log rows present).\n\n"
    "Notable observations from the aggregates:\n"
    "- metric and trace data are dense across the same short window (~06:37 to 07:01 UTC).\n"
    "- Metrics show multiple services with per-KPI data; examples include adservice, cartservice, checkoutservice.\n"
    "- Some metric p95 values are large for memory/diskio (e.g., very large mem/diskio p95 in metric groups),\n"
    "  and socket counts appear as non-zero for some services (socket p95 up to 9 in checkoutservice in the top groups).\n"
    "- Trace groups are dominated by checkoutservice traces (many trace.* fields present) with row_count and duration p95 values observed.\n"
    "- Log data shows log.row_count p95 high for frontend and currencyservice; log.error_count p95 is 0 across the top groups.\n"
    "- No error_logs rows were captured in error_logs.csv during this window.\n\n"
    "This is a compact summary of the available telemetry aggregates; full time-series rows are not shown here."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(total_rows                                 1850
earliest_timestamp    2024-01-20 06:37:00+00:00
latest_timestamp      2024-01-20 07:01:00+00:00
dtype: object, 0                 adservice
1               cartservice
2           checkoutservice
3           currencyservice
4              emailservice
5                  frontend
6         frontend-external
7            paymentservice
8     productcatalogservice
9     recommendationservice
10                    redis
11          shippingservice
dtype: object, 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
dtype: object,             cmdb_id    kpi_name  count           p95
0         adservice         cpu     25  7.982452e-01
1         adservice      diskio     25  1.023287e+05
2         adservice  latency-50     25  3.026546e-03
3         adservice  latency-90     25  4.654824e-03
4         adservice         mem     25  1.090296e+08
5         adservice      socket     25  4.000000e+00
6         adservice    workload     25  1.562652e+01
7       cartservice         cpu     25  2.713070e+00
8       cartservice      diskio     25  0.000000e+00
9       cartservice  latency-50     25  3.206973e-03
10      cartservice  latency-90     25  5.057485e-03
11      cartservice         mem     25  4.452464e+07
12      cartservice      socket     25  4.000000e+00
13      cartservice    workload     25  2.672056e+01
14  checkoutservice         cpu     25  4.974310e-01
15  checkoutservice  latency-50     25  3.377572e-01
16  checkoutservice  latency-90     25  4.680232e-01
17  checkoutservice         mem     25  1.195947e+07
18  checkoutservice      socket     25  9.000000e+00
19  checkoutservice    workload     25  1.143037e+00, total_rows                                 3784
earliest_timestamp    2024-01-20 06:37:00+00:00
latest_timestamp      2024-01-20 07:01:00+00:00
dtype: object, 0          checkoutservice
1          currencyservice
2             emailservice
3          frontendservice
4           paymentservice
5    productcatalogservice
6    recommendationservice
7                     root
dtype: object, 0         trace.from_checkoutservice.duration_mean
1          trace.from_checkoutservice.duration_p95
2            trace.from_checkoutservice.error_rate
3             trace.from_checkoutservice.row_count
4         trace.from_frontendservice.duration_mean
                          ...                     
39        trace.to_productcatalogservice.row_count
40    trace.to_recommendationservice.duration_mean
41     trace.to_recommendationservice.duration_p95
42       trace.to_recommendationservice.error_rate
43        trace.to_recommendationservice.row_count
Length: 44, dtype: object,             cmdb_id                                trace_name  count         p95
0   checkoutservice  trace.from_checkoutservice.duration_mean     25    0.030224
1   checkoutservice   trace.from_checkoutservice.duration_p95     25    0.204295
2   checkoutservice     trace.from_checkoutservice.error_rate     25    0.000000
3   checkoutservice      trace.from_checkoutservice.row_count     25  289.200000
4   checkoutservice  trace.from_frontendservice.duration_mean     25    0.301331
5   checkoutservice   trace.from_frontendservice.duration_p95     25    0.398125
6   checkoutservice     trace.from_frontendservice.error_rate     25    0.000000
7   checkoutservice      trace.from_frontendservice.row_count     25   34.600000
8   checkoutservice             trace.from_root.duration_mean     25    0.016882
9   checkoutservice              trace.from_root.duration_p95     25    0.072784
10  checkoutservice                trace.from_root.error_rate     25    0.000000
11  checkoutservice                 trace.from_root.row_count     25  140.800000
12  checkoutservice    trace.to_checkoutservice.duration_mean     25    0.030224
13  checkoutservice     trace.to_checkoutservice.duration_p95     25    0.204295
14  checkoutservice       trace.to_checkoutservice.error_rate     25    0.000000
15  checkoutservice        trace.to_checkoutservice.row_count     25  289.200000
16  checkoutservice    trace.to_currencyservice.duration_mean     25    0.002017
17  checkoutservice     trace.to_currencyservice.duration_p95     25    0.000276
18  checkoutservice       trace.to_currencyservice.error_rate     25    0.000000
19  checkoutservice        trace.to_currencyservice.row_count     25  116.800000, total_rows                                  458
earliest_timestamp    2024-01-20 06:37:00+00:00
latest_timestamp      2024-01-20 07:01:00+00:00
dtype: object, 0                adservice
1              cartservice
2          checkoutservice
3          currencyservice
4             emailservice
5                 frontend
6           paymentservice
7    recommendationservice
8                    redis
9          shippingservice
dtype: object, 0    log.error_count
1      log.row_count
dtype: object,                   cmdb_id         log_name  count     p95
0               adservice  log.error_count     25     0.0
1               adservice    log.row_count     25   461.8
2             cartservice  log.error_count     25     0.0
3             cartservice    log.row_count     25   804.4
4         checkoutservice  log.error_count     25     0.0
5         checkoutservice    log.row_count     25   103.8
6         currencyservice  log.error_count     25     0.0
7         currencyservice    log.row_count     25  2493.0
8            emailservice  log.error_count     25     0.0
9            emailservice    log.row_count     25    34.6
10               frontend  log.error_count     25     0.0
11               frontend    log.row_count     25  2771.4
12         paymentservice  log.error_count     25     0.0
13         paymentservice    log.row_count     25    69.2
14  recommendationservice  log.error_count     25     0.0
15  recommendationservice    log.row_count     25   579.8
19        shippingservice    log.row_count     25   510.0
18        shippingservice  log.error_count     25     0.0
17                  redis    log.row_count      4     5.0
16                  redis  log.error_count      4     0.0, 0, Series([], Name: count, dtype: int64), Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# reuse existing df_metric if present, otherwise load
if 'df_metric' not in globals():
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# ensure timestamp is parsed
if not pd.api.types.is_datetime64tz_dtype(df_metric['timestamp']):
    df_metric['timestamp'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) compute global P95 per (cmdb_id, kpi_name) using the full series
global_p95_df = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_p95')
)

# 2) filter metric.csv to the incident window (UTC)
start = pd.to_datetime('2024-01-20 06:34:06', utc=True)
end   = pd.to_datetime('2024-01-20 07:04:06', utc=True)
df_window = df_metric[(df_metric['timestamp'] >= start) & (df_metric['timestamp'] <= end)].copy()

# merge global_p95 into windowed rows for anomaly detection
df_window = df_window.merge(global_p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# mark anomalies where value >= global_p95
df_window['is_anomaly'] = df_window['value'] >= df_window['global_p95']

# aggregated stats in-window per (cmdb_id, kpi_name)
window_stats = (
    df_window
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        count_in_window=('value', 'count'),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# aggregated anomaly stats in-window per (cmdb_id, kpi_name)
anomaly_stats = (
    df_window[df_window['is_anomaly']]
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .agg(
        anomaly_count=('value', 'count'),
        earliest_anomaly_ts=('timestamp', 'min')
    )
    .reset_index()
)

# combine with global_p95 to ensure we include groups with zero anomalies
summary = (
    global_p95_df
    .merge(window_stats, on=['cmdb_id', 'kpi_name'], how='left')
    .merge(anomaly_stats, on=['cmdb_id', 'kpi_name'], how='left')
)

# fill missing values: counts -> 0, earliest_anomaly_ts -> None (null), keep max_value_in_window as NaN if no rows
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
# earliest_anomaly_ts keep as datetime or None
summary['earliest_anomaly_ts'] = pd.to_datetime(summary['earliest_anomaly_ts'], utc=True)
# format earliest_anomaly_ts as ISO UTC string where present, else None
summary['earliest_anomaly_ts_iso'] = summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary.loc[summary['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# select and rename output columns
out_cols = ['cmdb_id', 'kpi_name', 'anomaly_count', 'earliest_anomaly_ts_iso', 'max_value_in_window', 'global_p95', 'count_in_window']
summary_table = summary[out_cols].copy()
summary_table = summary_table.rename(columns={'earliest_anomaly_ts_iso': 'earliest_anomaly_ts'})

# sort by anomaly_count desc, then earliest_anomaly_ts asc (nulls last)
summary_table['earliest_sort'] = pd.to_datetime(summary_table['earliest_anomaly_ts'], utc=True)
summary_table = summary_table.sort_values(by=['anomaly_count', 'earliest_sort'], ascending=[False, True])
summary_table = summary_table.drop(columns=['earliest_sort'])

# limit to top 20 rows as requested
summary_table_top20 = summary_table.head(20)

# prepare anomaly rows for the specific component-KPI checkoutservice/socket
anomaly_rows = df_window[df_window['is_anomaly']].copy()
checkout_socket_anomalies = anomaly_rows[
    (anomaly_rows['cmdb_id'] == 'checkoutservice') & (anomaly_rows['kpi_name'] == 'socket')
].sort_values('timestamp')[['timestamp', 'cmdb_id', 'kpi_name', 'value']]

# limit to top 20 rows
checkout_socket_anomalies_top20 = checkout_socket_anomalies.head(20)

# Display compact results: summary_table_top20 and checkout_socket_anomalies_top20
summary_table_top20, checkout_socket_anomalies_top20

```
Out[2]:
```
```python
# Plain-English summary of the analysis results (compact)
summary = (
    "Summary of metric anomaly analysis (UTC):\n\n"
    "• Global P95s were computed per (cmdb_id, kpi_name) using the full metric series.\n"
    "• During the incident window 2024-01-20 06:34:06 to 2024-01-20 07:04:06 UTC many (cmdb_id, kpi_name) pairs had values >= their global P95 repeatedly.\n"
    "• checkoutservice/socket: anomaly_count = 25 (every sampled row in-window met or exceeded global P95). "
    "Earliest anomaly = 2024-01-20T06:37:00Z. max_value_in_window = 9.0. global_P95 = 9.0. "
    "The sampled anomaly rows show sustained per-minute values of 9.0 starting at 06:37Z.\n"
    "• Multiple other services show the same pattern for the 'socket' KPI (adservice, cartservice, currencyservice, emailservice, "
    "productcatalogservice, recommendationservice, redis, shippingservice) with anomaly_count = 25 and socket values equal to their global P95s.\n"
    "• Several services have diskio and error KPIs with global P95 = 0 and matching in-window values (thus counted as anomalies by the >= rule).\n\n"
    "Interpretation: the socket KPI reached its global P95 across many services and was persistent for checkoutservice (and others) during the window, "
    "indicating a sustained, system-wide elevation of the socket metric rather than a single short spike.\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id kpi_name  anomaly_count   earliest_anomaly_ts  max_value_in_window    global_p95  count_in_window
5               adservice   socket             25  2024-01-20T06:37:00Z         4.000000e+00  4.000000e+00               25
8             cartservice   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
12            cartservice   socket             25  2024-01-20T06:37:00Z         4.000000e+00  4.000000e+00               25
18        checkoutservice   socket             25  2024-01-20T06:37:00Z         9.000000e+00  9.000000e+00               25
21        currencyservice   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
25        currencyservice   socket             25  2024-01-20T06:37:00Z         3.000000e+00  3.000000e+00               25
28           emailservice   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
32           emailservice   socket             25  2024-01-20T06:37:00Z         3.000000e+00  3.000000e+00               25
35               frontend   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
36               frontend    error             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
42      frontend-external    error             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
48         paymentservice   socket             25  2024-01-20T06:37:00Z         3.000000e+00  3.000000e+00               25
51  productcatalogservice   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
55  productcatalogservice   socket             25  2024-01-20T06:37:00Z         3.000000e+00  3.000000e+00               25
58  recommendationservice   diskio             25  2024-01-20T06:37:00Z         0.000000e+00  0.000000e+00               25
62  recommendationservice   socket             25  2024-01-20T06:37:00Z         4.000000e+00  4.000000e+00               25
67                  redis   socket             25  2024-01-20T06:37:00Z         4.000000e+00  4.000000e+00               25
72        shippingservice   socket             25  2024-01-20T06:37:00Z         3.000000e+00  3.000000e+00               25
40               frontend   socket             16  2024-01-20T06:46:00Z         2.600000e+01  2.600000e+01               25
66                  redis      mem              2  2024-01-20T06:37:00Z         3.022575e+06  3.005634e+06               25,                      timestamp          cmdb_id kpi_name  value
18   2024-01-20 06:37:00+00:00  checkoutservice   socket    9.0
92   2024-01-20 06:38:00+00:00  checkoutservice   socket    9.0
166  2024-01-20 06:39:00+00:00  checkoutservice   socket    9.0
240  2024-01-20 06:40:00+00:00  checkoutservice   socket    9.0
314  2024-01-20 06:41:00+00:00  checkoutservice   socket    9.0
388  2024-01-20 06:42:00+00:00  checkoutservice   socket    9.0
462  2024-01-20 06:43:00+00:00  checkoutservice   socket    9.0
536  2024-01-20 06:44:00+00:00  checkoutservice   socket    9.0
610  2024-01-20 06:45:00+00:00  checkoutservice   socket    9.0
684  2024-01-20 06:46:00+00:00  checkoutservice   socket    9.0
758  2024-01-20 06:47:00+00:00  checkoutservice   socket    9.0
832  2024-01-20 06:48:00+00:00  checkoutservice   socket    9.0
906  2024-01-20 06:49:00+00:00  checkoutservice   socket    9.0
980  2024-01-20 06:50:00+00:00  checkoutservice   socket    9.0
1054 2024-01-20 06:51:00+00:00  checkoutservice   socket    9.0
1128 2024-01-20 06:52:00+00:00  checkoutservice   socket    9.0
1202 2024-01-20 06:53:00+00:00  checkoutservice   socket    9.0
1276 2024-01-20 06:54:00+00:00  checkoutservice   socket    9.0
1350 2024-01-20 06:55:00+00:00  checkoutservice   socket    9.0
1424 2024-01-20 06:56:00+00:00  checkoutservice   socket    9.0)```
```

In [ ]:
# In[3]:

import pandas as pd

# reuse df_trace if available, else load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp is datetime tz-aware
    if not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# 1) compute global P95 and P05 per (cmdb_id, trace_name) using the full series
global_q = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(global_P95=lambda x: float(x.quantile(0.95)),
         global_P05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) filter to the incident window (UTC)
start = pd.to_datetime('2024-01-20 06:34:06', utc=True)
end   = pd.to_datetime('2024-01-20 07:04:06', utc=True)
df_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# merge global thresholds into windowed rows
df_window = df_window.merge(global_q, on=['cmdb_id', 'trace_name'], how='left')

# 3) anomaly condition per row:
# - value >= global_P95
# - OR (trace_name contains 'error_rate' and value > 0)
# - OR (trace_name contains 'row_count' and value <= global_P05)
contains_error = df_window['trace_name'].str.contains('error_rate', na=False)
contains_rowcount = df_window['trace_name'].str.contains('row_count', na=False)

cond_p95 = df_window['value'] >= df_window['global_P95']
cond_error = contains_error & (df_window['value'] > 0)
cond_rowcount = contains_rowcount & (df_window['value'] <= df_window['global_P05'])

df_window['is_anomaly'] = cond_p95 | cond_error | cond_rowcount

# 4) aggregate per (cmdb_id, trace_name)
# anomaly counts and earliest anomaly timestamp
anomaly_agg = (
    df_window[df_window['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(anomaly_count=('value', 'count'),
         earliest_anomaly_ts=('timestamp', 'min'))
    .reset_index()
)

# min and max value in window (for groups that may have no rows, these will be NaN)
minmax_agg = (
    df_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(min_value_in_window=('value', 'min'),
         max_value_in_window=('value', 'max'),
         count_in_window=('value', 'count'))
    .reset_index()
)

# combine all into a single summary, starting from global_q to include all series
summary = (
    global_q
    .merge(minmax_agg, on=['cmdb_id', 'trace_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'trace_name'], how='left')
)

# fill missing anomaly_count with 0
summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)

# format earliest_anomaly_ts as ISO UTC string or None
summary['earliest_anomaly_ts'] = pd.to_datetime(summary['earliest_anomaly_ts'], utc=True)
summary['earliest_anomaly_ts_iso'] = summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary.loc[summary['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# prepare final columns
out_cols = [
    'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'min_value_in_window', 'max_value_in_window', 'global_P95', 'global_P05'
]
summary_table = summary[out_cols].rename(columns={'earliest_anomaly_ts_iso': 'earliest_anomaly_ts'})

# sort by anomaly_count desc, then earliest_anomaly_ts asc (nulls last)
# create sort key for earliest where None -> max timestamp
earliest_sort = pd.to_datetime(summary_table['earliest_anomaly_ts'], utc=True)
earliest_sort = earliest_sort.fillna(pd.Timestamp.max.tz_localize('UTC'))
summary_table['__earliest_sort'] = earliest_sort

summary_table_sorted = summary_table.sort_values(by=['anomaly_count', '__earliest_sort'], ascending=[False, True]).drop(columns=['__earliest_sort'])

# limit to top 20 rows for compact display
trace_summary_top20 = summary_table_sorted.head(20)

# 5) prepare checkoutservice in-window trace records (timestamp, trace_name, value), up to 50 rows
checkout_window = df_window[df_window['cmdb_id'] == 'checkoutservice'][['timestamp', 'trace_name', 'value']].sort_values('timestamp').head(50)

# Display compact outputs
trace_summary_top20, checkout_window

```
Out[3]:
```
```python
# Plain-English summary of the trace anomaly results (compact)
final_summary = (
    "Trace anomaly summary for 2024-01-20 06:34:06 - 07:04:06 UTC:\n\n"
    "• Many (cmdb_id, trace_name) pairs show anomaly_count = 25 with earliest anomaly at 2024-01-20T06:37:00Z. "
    "These are dominated by 'error_rate' series across services (including checkoutservice) that have value == 0.0 and global_P95 == 0.0. "
    "Because the rule flagged rows with value >= global_P95, all-zero series were counted as anomalies (a thresholding artifact).\n\n"
    "• For checkoutservice specifically: multiple error_rate traces (from_checkoutservice, from_frontendservice, from_root, to_checkoutservice, "
    "to_currencyservice, to_emailservice, to_paymentservice, to_productcatalogservice) have anomaly_count = 25, min=max=0.0, global_P95=0.0, global_P05=0.0. "
    "This indicates no non-zero error rates were observed in-window; they were flagged only because the P95 threshold equals 0.\n\n"
    "• Some row_count traces show meaningful signals: e.g., emailservice.trace.from_root.row_count has min=2, max=24, global_P95=24, global_P05=22.4 and was flagged (values <= P05). "
    "Row_count anomalies reflect unusually low counts in-window relative to historical low quantile.\n\n"
    "• The returned checkoutservice in-window trace rows (sample) show a mix of small latency values and row_count entries (for example, a trace.to_productcatalogservice.row_count = 58 at 06:37), "
    "but no positive error_rate values—so no direct evidence of increased trace error rates for checkoutservice during the window.\n\n"
    "Interpretation and recommendation:\n"
    "- The anomaly list is largely driven by thresholding edge cases (global_P95 or P05 equal to 0), producing many false-positive flags for zero-valued series. "
    "- Re-evaluate detection rules: avoid treating value >= global_P95 as an anomaly when global_P95 == 0 for mostly-zero signals; consider using strict > for P95, require non-zero baselines, or require change relative to recent baseline. "
    "- Focus investigation on series with non-zero error_rate or significant deviations in row_count/latency; specifically review any services where row_count is persistently below historical P05 or where latency/row_count exceed meaningful P95 thresholds.\n"
)
final_summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                                 trace_name  anomaly_count   earliest_anomaly_ts  min_value_in_window  max_value_in_window  global_P95  global_P05
2   checkoutservice      trace.from_checkoutservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
6   checkoutservice      trace.from_frontendservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
10  checkoutservice                 trace.from_root.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
14  checkoutservice        trace.to_checkoutservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
18  checkoutservice        trace.to_currencyservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
22  checkoutservice           trace.to_emailservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
26  checkoutservice         trace.to_paymentservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
30  checkoutservice  trace.to_productcatalogservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
34  currencyservice      trace.from_checkoutservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
38  currencyservice      trace.from_frontendservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
46     emailservice      trace.from_checkoutservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
50     emailservice                 trace.from_root.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
51     emailservice                  trace.from_root.row_count             25  2024-01-20T06:37:00Z                  2.0                 24.0        24.0        22.4
54  frontendservice      trace.from_frontendservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
58  frontendservice                 trace.from_root.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
62  frontendservice        trace.to_checkoutservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
66  frontendservice        trace.to_currencyservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
70  frontendservice        trace.to_frontendservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
74  frontendservice  trace.to_productcatalogservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0
78  frontendservice  trace.to_recommendationservice.error_rate             25  2024-01-20T06:37:00Z                  0.0                  0.0         0.0         0.0,                     timestamp                                    trace_name      value
0   2024-01-20 06:37:00+00:00      trace.from_checkoutservice.duration_mean   0.003795
31  2024-01-20 06:37:00+00:00      trace.to_productcatalogservice.row_count  58.000000
30  2024-01-20 06:37:00+00:00     trace.to_productcatalogservice.error_rate   0.000000
29  2024-01-20 06:37:00+00:00   trace.to_productcatalogservice.duration_p95   0.000033
28  2024-01-20 06:37:00+00:00  trace.to_productcatalogservice.duration_mean   0.000023
..                        ...                                           ...        ...
177 2024-01-20 06:38:00+00:00          trace.to_paymentservice.duration_p95   0.000447
168 2024-01-20 06:38:00+00:00        trace.to_currencyservice.duration_mean   0.001525
175 2024-01-20 06:38:00+00:00               trace.to_emailservice.row_count  32.000000
166 2024-01-20 06:38:00+00:00           trace.to_checkoutservice.error_rate   0.000000
153 2024-01-20 06:38:00+00:00       trace.from_checkoutservice.duration_p95   0.005378

[50 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse df_log and df_error if present, else load
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')
    df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64tz_dtype(df_log['timestamp']):
        df_log['timestamp'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

if 'df_error' not in globals():
    df_error = pd.read_csv('error_logs.csv')
    df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64tz_dtype(df_error['timestamp']):
        df_error['timestamp'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, log_name) using full series
log_thresholds = (
    df_log
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .agg(global_P95=lambda x: float(x.quantile(0.95)),
         global_P05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to incident window (UTC)
start = pd.to_datetime('2024-01-20 06:34:06', utc=True)
end   = pd.to_datetime('2024-01-20 07:04:06', utc=True)
df_log_window = df_log[(df_log['timestamp'] >= start) & (df_log['timestamp'] <= end)].copy()

# 3) Merge thresholds into windowed rows
df_log_window = df_log_window.merge(log_thresholds, on=['cmdb_id', 'log_name'], how='left')

# 4) Define anomaly condition per rules:
# - if 'error' in log_name -> value > 0
# - elif 'row_count' in log_name -> value <= global_P05
# - else -> value >= global_P95
contains_error = df_log_window['log_name'].str.contains('error', na=False)
contains_rowcount = df_log_window['log_name'].str.contains('row_count', na=False)

cond_error = contains_error & (df_log_window['value'] > 0)
cond_rowcount = contains_rowcount & (df_log_window['value'] <= df_log_window['global_P05'])
cond_other = (~contains_error & ~contains_rowcount) & (df_log_window['value'] >= df_log_window['global_P95'])

df_log_window['is_anomaly'] = cond_error | cond_rowcount | cond_other

# 5) Aggregate anomaly stats and min/max in-window
anomaly_agg = (
    df_log_window[df_log_window['is_anomaly']]
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(anomaly_count=('value', 'count'),
         earliest_anomaly_ts=('timestamp', 'min'))
    .reset_index()
)

minmax_agg = (
    df_log_window
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .agg(min_value_in_window=('value', 'min'),
         max_value_in_window=('value', 'max'),
         count_in_window=('value', 'count'))
    .reset_index()
)

# 6) Combine with global thresholds to produce final summary
log_summary = (
    log_thresholds
    .merge(minmax_agg, on=['cmdb_id', 'log_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'log_name'], how='left')
)

log_summary['anomaly_count'] = log_summary['anomaly_count'].fillna(0).astype(int)
log_summary['count_in_window'] = log_summary['count_in_window'].fillna(0).astype(int)

log_summary['earliest_anomaly_ts'] = pd.to_datetime(log_summary['earliest_anomaly_ts'], utc=True)
log_summary['earliest_anomaly_ts_iso'] = log_summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
log_summary.loc[log_summary['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

out_cols = [
    'cmdb_id', 'log_name', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'min_value_in_window', 'max_value_in_window', 'global_P95', 'global_P05'
]
log_summary_table = log_summary[out_cols].rename(columns={'earliest_anomaly_ts_iso': 'earliest_anomaly_ts'})

# sort by anomaly_count desc, then earliest_anomaly_ts asc (nulls last)
earliest_sort = pd.to_datetime(log_summary_table['earliest_anomaly_ts'], utc=True)
earliest_sort = earliest_sort.fillna(pd.Timestamp.max.tz_localize('UTC'))
log_summary_table['__earliest_sort'] = earliest_sort
log_summary_sorted = log_summary_table.sort_values(by=['anomaly_count', '__earliest_sort'], ascending=[False, True]).drop(columns=['__earliest_sort'])

log_summary_top20 = log_summary_sorted.head(20)

# 7) Up to 20 raw log rows in-window for checkoutservice
checkout_logs_window = df_log_window[df_log_window['cmdb_id'] == 'checkoutservice'][['timestamp', 'cmdb_id', 'log_name', 'value']].sort_values('timestamp').head(20)

# 8) error_logs.csv total count and up to 5 sample rows sorted by timestamp asc
error_total_rows = len(df_error)
error_sample_rows = df_error.sort_values('timestamp')[['timestamp', 'cmdb_id', 'message']].head(5)

# Display compact outputs (variables for IPython)
log_summary_top20, checkout_logs_window, error_total_rows, error_sample_rows

```
Out[4]:
```
```python
summary = (
    "Log analysis summary (UTC):\n\n"
    "• Global P95/P05 were computed per (cmdb_id, log_name) using the full series.\n\n"
    "• In the incident window 2024-01-20 06:34:06 – 07:04:06 UTC:\n"
    "  - Many services show anomalies on log.row_count (anomaly_count = 2 for adservice, cartservice, checkoutservice, currencyservice, "
    "emailservice, frontend, paymentservice, recommendationservice, shippingservice). Earliest anomaly for these groups is 2024-01-20T06:37:00Z.\n"
    "  - Redis.log.row_count is anomalous 4 times in-window (earliest 2024-01-20T06:41:00Z); min=max in-window = 5, global_P95 = 5.0, global_P05 = 5.0.\n\n"
    "• checkoutservice specifics:\n"
    "  - log.row_count: anomaly_count = 2, earliest_anomaly = 2024-01-20T06:37:00Z.\n"
    "    min_value_in_window = 18, max_value_in_window = 113, global_P95 = 103.8, global_P05 = 56.0.\n"
    "  - The raw checkoutservice log rows in-window show alternating log.error_count = 0 and log.row_count values per minute (examples: row_count values observed include 55, 96, 89, 63, 99, 81, 60, 72, 81, 105). "
    "There are no non-zero error_count entries for checkoutservice in-window.\n\n"
    "• log.error_count: anomaly_count = 0 for all services (all observed values = 0 and global_P95 = 0.0), so no error spikes were detected in logs.\n\n"
    "• error_logs.csv: total rows = 0 (no error log samples present).\n\n"
    "Interpretation:\n"
    "- There were no increases in logged errors. The notable signals are low row_count events (values <= historical P05) for several services, including checkoutservice (two low samples) and a small/constant row_count for redis. "
    "- These row_count anomalies indicate reduced per-minute log volume for those services during the window, not increased error counts. Focus investigation on why request/row counts dipped for affected services rather than on log error spikes."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id         log_name  anomaly_count   earliest_anomaly_ts  min_value_in_window  max_value_in_window  global_P95  global_P05
17                  redis    log.row_count              4  2024-01-20T06:41:00Z                    5                    5         5.0         5.0
1               adservice    log.row_count              2  2024-01-20T06:37:00Z                   51                  475       461.8       391.2
3             cartservice    log.row_count              2  2024-01-20T06:37:00Z                   96                  807       804.4       640.0
5         checkoutservice    log.row_count              2  2024-01-20T06:37:00Z                   18                  113       103.8        56.0
7         currencyservice    log.row_count              2  2024-01-20T06:37:00Z                  297                 2523      2493.0      2037.0
9            emailservice    log.row_count              2  2024-01-20T06:37:00Z                    6                   37        34.6        18.4
11               frontend    log.row_count              2  2024-01-20T06:37:00Z                  328                 2780      2771.4      2255.8
13         paymentservice    log.row_count              2  2024-01-20T06:37:00Z                   12                   76        69.2        36.8
15  recommendationservice    log.row_count              2  2024-01-20T06:37:00Z                   70                  586       579.8       479.6
19        shippingservice    log.row_count              2  2024-01-20T06:37:00Z                   70                  526       510.0       360.4
0               adservice  log.error_count              0                  None                    0                    0         0.0         0.0
2             cartservice  log.error_count              0                  None                    0                    0         0.0         0.0
4         checkoutservice  log.error_count              0                  None                    0                    0         0.0         0.0
6         currencyservice  log.error_count              0                  None                    0                    0         0.0         0.0
8            emailservice  log.error_count              0                  None                    0                    0         0.0         0.0
10               frontend  log.error_count              0                  None                    0                    0         0.0         0.0
12         paymentservice  log.error_count              0                  None                    0                    0         0.0         0.0
14  recommendationservice  log.error_count              0                  None                    0                    0         0.0         0.0
16                  redis  log.error_count              0                  None                    0                    0         0.0         0.0
18        shippingservice  log.error_count              0                  None                    0                    0         0.0         0.0,                     timestamp          cmdb_id         log_name  value
4   2024-01-20 06:37:00+00:00  checkoutservice  log.error_count      0
5   2024-01-20 06:37:00+00:00  checkoutservice    log.row_count     55
22  2024-01-20 06:38:00+00:00  checkoutservice  log.error_count      0
23  2024-01-20 06:38:00+00:00  checkoutservice    log.row_count     96
40  2024-01-20 06:39:00+00:00  checkoutservice  log.error_count      0
41  2024-01-20 06:39:00+00:00  checkoutservice    log.row_count     89
58  2024-01-20 06:40:00+00:00  checkoutservice  log.error_count      0
59  2024-01-20 06:40:00+00:00  checkoutservice    log.row_count     63
76  2024-01-20 06:41:00+00:00  checkoutservice  log.error_count      0
77  2024-01-20 06:41:00+00:00  checkoutservice    log.row_count     99
96  2024-01-20 06:42:00+00:00  checkoutservice  log.error_count      0
97  2024-01-20 06:42:00+00:00  checkoutservice    log.row_count     81
115 2024-01-20 06:43:00+00:00  checkoutservice    log.row_count     60
114 2024-01-20 06:43:00+00:00  checkoutservice  log.error_count      0
132 2024-01-20 06:44:00+00:00  checkoutservice  log.error_count      0
133 2024-01-20 06:44:00+00:00  checkoutservice    log.row_count     72
150 2024-01-20 06:45:00+00:00  checkoutservice  log.error_count      0
151 2024-01-20 06:45:00+00:00  checkoutservice    log.row_count     81
168 2024-01-20 06:46:00+00:00  checkoutservice  log.error_count      0
169 2024-01-20 06:46:00+00:00  checkoutservice    log.row_count    105, 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])```
```

In [ ]:
# In[5]:

import pandas as pd

# reuse df_trace if present, otherwise load and parse timestamps
if 'df_trace' not in globals():
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
else:
    if not pd.api.types.is_datetime64tz_dtype(df_trace['timestamp']):
        df_trace['timestamp'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# 1) Compute global P95 and P05 per (cmdb_id, trace_name) using the full trace series
global_q = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(global_P95=lambda x: float(x.quantile(0.95)),
         global_P05=lambda x: float(x.quantile(0.05)))
    .reset_index()
)

# 2) Filter to the incident window (UTC)
start = pd.to_datetime('2024-01-20 06:34:06', utc=True)
end   = pd.to_datetime('2024-01-20 07:04:06', utc=True)
df_window = df_trace[(df_trace['timestamp'] >= start) & (df_trace['timestamp'] <= end)].copy()

# 3) Merge global thresholds into windowed rows
df_window = df_window.merge(global_q, on=['cmdb_id', 'trace_name'], how='left')

# 4) Define anomaly conditions per rules:
# - duration (contains 'duration'): anomaly if value >= global_P95 and global_P95 > 0
# - error_rate: anomaly if value > 0
# - row_count: anomaly if value <= global_P05 and global_P05 < global_P95 (i.e., variability)
contains_duration = df_window['trace_name'].str.contains('duration', na=False)
contains_error = df_window['trace_name'].str.contains('error_rate', na=False)
contains_rowcount = df_window['trace_name'].str.contains('row_count', na=False)

cond_duration = contains_duration & (df_window['global_P95'] > 0) & (df_window['value'] >= df_window['global_P95'])
cond_error = contains_error & (df_window['value'] > 0)
cond_rowcount = contains_rowcount & (df_window['global_P05'] < df_window['global_P95']) & (df_window['value'] <= df_window['global_P05'])

df_window['is_anomaly'] = cond_duration | cond_error | cond_rowcount

# 5) Aggregate per (cmdb_id, trace_name)
# anomaly counts and earliest anomaly timestamp
anomaly_agg = (
    df_window[df_window['is_anomaly']]
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(anomaly_count=('value', 'count'),
         earliest_anomaly_ts=('timestamp', 'min'))
    .reset_index()
)

# min and max value in window and counts
minmax_agg = (
    df_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .agg(min_value_in_window=('value', 'min'),
         max_value_in_window=('value', 'max'),
         count_in_window=('value', 'count'))
    .reset_index()
)

# 6) Combine into summary starting from global_q to include all series
summary = (
    global_q
    .merge(minmax_agg, on=['cmdb_id', 'trace_name'], how='left')
    .merge(anomaly_agg, on=['cmdb_id', 'trace_name'], how='left')
)

summary['anomaly_count'] = summary['anomaly_count'].fillna(0).astype(int)
summary['count_in_window'] = summary['count_in_window'].fillna(0).astype(int)

# format earliest_anomaly_ts as ISO UTC string or None
summary['earliest_anomaly_ts'] = pd.to_datetime(summary['earliest_anomaly_ts'], utc=True)
summary['earliest_anomaly_ts_iso'] = summary['earliest_anomaly_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
summary.loc[summary['earliest_anomaly_ts'].isna(), 'earliest_anomaly_ts_iso'] = None

# select requested output columns
out_cols = [
    'cmdb_id', 'trace_name', 'anomaly_count', 'earliest_anomaly_ts_iso',
    'min_value_in_window', 'max_value_in_window', 'global_P95', 'global_P05'
]
trace_summary_table = summary[out_cols].rename(columns={'earliest_anomaly_ts_iso': 'earliest_anomaly_ts'})

# sort by anomaly_count desc, then earliest_anomaly_ts asc (nulls last)
earliest_sort = pd.to_datetime(trace_summary_table['earliest_anomaly_ts'], utc=True)
earliest_sort = earliest_sort.fillna(pd.Timestamp.max.tz_localize('UTC'))
trace_summary_table['__earliest_sort'] = earliest_sort
trace_summary_sorted = trace_summary_table.sort_values(by=['anomaly_count', '__earliest_sort'], ascending=[False, True]).drop(columns=['__earliest_sort'])

# limit to top 20 rows for compactness
trace_summary_top20 = trace_summary_sorted.head(20)

# 7) Prepare checkoutservice in-window trace records where trace_name contains 'to_' or 'from_' or 'row_count' or 'duration'
pattern = r'(^.*\bto_|\bfrom_|row_count|duration)'
checkout_selected_window = (
    df_window[
        (df_window['cmdb_id'] == 'checkoutservice') &
        (df_window['trace_name'].str.contains('to_|from_|row_count|duration', na=False))
    ][['timestamp', 'trace_name', 'value']]
    .sort_values('timestamp')
    .head(50)
)

# Display compact outputs (variables for IPython)
trace_summary_top20, checkout_selected_window

```
Out[5]:
```
```python
summary = (
    "Trace anomaly summary (UTC):\n\n"
    "• Main findings:\n"
    "  - The strongest anomalies are low row_count events involving productcatalogservice and calls to it. "
    "trace.to_productcatalogservice.row_count (checkoutservice) and trace.from_checkoutservice.row_count (productcatalogservice) each have anomaly_count=3 with earliest anomaly at 2024-01-20T06:43:00Z (in-window min ~9, max ~84, global_P95 ~83.4, global_P05 ~50). "
    "This indicates a brief drop in observed row_count to well below the historical low quantile around 06:43.\n"
    "  - There are duration_p95 anomalies around 06:47 affecting calls to/from productcatalogservice (e.g., frontendservice trace.to_productcatalogservice.duration_p95 and its productcatalogservice counterpart, anomaly_count=3, global_P95 ~2.9e-05). "
    "This suggests a small increase in tail latency near 06:47 for productcatalogservice-related calls.\n"
    "  - Checkoutservice shows multiple row_count anomalies earlier at ~06:37 (several trace.*.row_count entries with anomaly_count=2) and some duration anomalies for calls to currencyservice (duration_mean/p95 flagged). "
    "Error_rate series remain zero (no positive error_rate spikes).\n\n"
    "• Interpretation and next steps:\n"
    "  - The timeline suggests a transient reduction in traffic/hits to productcatalogservice around 06:43, followed by slight tail-latency elevation around 06:47. "
    "  - Checkoutservice experienced reduced row_counts around 06:37 and some duration deviations to currencyservice; however no error_rate increase was observed.\n"
    "  - Investigate productcatalogservice (resource usage, upstream throttling, network issues) and the specific call chain from checkoutservice/frontend to productcatalogservice around 06:43–06:47. "
    "Check service logs, deployment events, and any network/infra events at those times.\n\n"
    "Note: many error_rate series are zero and some zero-threshold edge cases were excluded by the applied rules; focus on non-zero error_rate, meaningful duration increases, and row_count drops as the most actionable signals."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                   trace_name  anomaly_count   earliest_anomaly_ts  min_value_in_window  max_value_in_window   global_P95   global_P05
31        checkoutservice     trace.to_productcatalogservice.row_count              3  2024-01-20T06:43:00Z             9.000000            84.000000    83.400000    50.000000
91  productcatalogservice         trace.from_checkoutservice.row_count              3  2024-01-20T06:43:00Z             9.000000            84.000000    83.400000    50.000000
73        frontendservice  trace.to_productcatalogservice.duration_p95              3  2024-01-20T06:47:00Z             0.000025             0.000029     0.000029     0.000025
93  productcatalogservice      trace.from_frontendservice.duration_p95              3  2024-01-20T06:47:00Z             0.000025             0.000029     0.000029     0.000025
3         checkoutservice         trace.from_checkoutservice.row_count              2  2024-01-20T06:37:00Z            45.000000           312.000000   289.200000   168.400000
7         checkoutservice         trace.from_frontendservice.row_count              2  2024-01-20T06:37:00Z             6.000000            38.000000    34.600000    19.200000
15        checkoutservice           trace.to_checkoutservice.row_count              2  2024-01-20T06:37:00Z            45.000000           312.000000   289.200000   168.400000
16        checkoutservice       trace.to_currencyservice.duration_mean              2  2024-01-20T06:37:00Z             0.000149             0.002254     0.002017     0.000151
17        checkoutservice        trace.to_currencyservice.duration_p95              2  2024-01-20T06:37:00Z             0.000195             0.000279     0.000276     0.000198
23        checkoutservice              trace.to_emailservice.row_count              2  2024-01-20T06:37:00Z             6.000000            37.000000    34.600000    18.400000
27        checkoutservice            trace.to_paymentservice.row_count              2  2024-01-20T06:37:00Z             6.000000            38.000000    34.600000    18.400000
32        currencyservice     trace.from_checkoutservice.duration_mean              2  2024-01-20T06:37:00Z             0.000149             0.002254     0.002017     0.000151
33        currencyservice      trace.from_checkoutservice.duration_p95              2  2024-01-20T06:37:00Z             0.000195             0.000279     0.000276     0.000198
39        currencyservice         trace.from_frontendservice.row_count              2  2024-01-20T06:37:00Z           282.000000          2421.000000  2388.600000  1962.200000
47           emailservice         trace.from_checkoutservice.row_count              2  2024-01-20T06:37:00Z             6.000000            37.000000    34.600000    18.400000
51           emailservice                    trace.from_root.row_count              2  2024-01-20T06:37:00Z             2.000000            24.000000    24.000000    22.400000
55        frontendservice         trace.from_frontendservice.row_count              2  2024-01-20T06:37:00Z           974.000000          8175.000000  8047.600000  6672.200000
59        frontendservice                    trace.from_root.row_count              2  2024-01-20T06:37:00Z           100.000000           847.000000   846.000000   690.000000
63        frontendservice           trace.to_checkoutservice.row_count              2  2024-01-20T06:37:00Z             6.000000            38.000000    34.600000    19.200000
67        frontendservice           trace.to_currencyservice.row_count              2  2024-01-20T06:37:00Z           282.000000          2421.000000  2388.600000  1962.200000,                     timestamp                                    trace_name      value
0   2024-01-20 06:37:00+00:00      trace.from_checkoutservice.duration_mean   0.003795
31  2024-01-20 06:37:00+00:00      trace.to_productcatalogservice.row_count  58.000000
30  2024-01-20 06:37:00+00:00     trace.to_productcatalogservice.error_rate   0.000000
29  2024-01-20 06:37:00+00:00   trace.to_productcatalogservice.duration_p95   0.000033
28  2024-01-20 06:37:00+00:00  trace.to_productcatalogservice.duration_mean   0.000023
..                        ...                                           ...        ...
177 2024-01-20 06:38:00+00:00          trace.to_paymentservice.duration_p95   0.000447
168 2024-01-20 06:38:00+00:00        trace.to_currencyservice.duration_mean   0.001525
175 2024-01-20 06:38:00+00:00               trace.to_emailservice.row_count  32.000000
166 2024-01-20 06:38:00+00:00           trace.to_checkoutservice.error_rate   0.000000
153 2024-01-20 06:38:00+00:00       trace.from_checkoutservice.duration_p95   0.005378

[50 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```